In [ ]:
import functools
import numpy as np
import qiskit as qk
import qiskit_aer
import qiskit.providers.fake_provider as qk_fp
import qiskit_dynamics as qk_d
from scipy.integrate import simpson
import qutip as qt

import sys
sys.path.append("../")
import pulse_simulator as ps

import matplotlib.pyplot as plt

In [ ]:
def get_closest_multiple_of(value, base_number):
    return int(value + base_number/2) - (int(value + base_number/2) % base_number)


def get_closest_multiple_of_16(num, granularity=16):
    return get_closest_multiple_of(num, granularity)


def qiskit_closest_multiple_of_16(gate_time_si_units, backend):
    with qk.pulse.build(name="dummy", backend=backend) as dummy:
        samples = get_closest_multiple_of_16(
            qk.pulse.seconds_to_samples(GATE_TIME_SI_UNITS)
        )
    return samples

In [ ]:
Units = 1e9

# Undo units
GHz = 1 / Units
MHz = 1e3 / Units
μs = Units / 1e3
ns = Units

backend = qk_fp.FakeManila()

dt = backend.configuration().dt * ns
desired_duration = 225 # dt
# Can be acquired with backend.configuration().timing_constraints['granularity']
granularity = 16
GATE_TIME_SI_UNITS = desired_duration * dt / ns
duration = qiskit_closest_multiple_of_16(GATE_TIME_SI_UNITS, backend)

# Load custom pulses

In [ ]:
pico_a = ps.from_csv("../pico-pulses/saved-pulses-2024-01-24/a_single_qubit_gateset_R1e-3.csv")
fig, ax = plt.subplots(figsize=[5,3])
for row in pico_a:
    ax.plot(row)

In [ ]:
np.trapz(pico_a[0], dx=dt) / np.pi

In [ ]:
def gateset_from_pico(duration, dt, pico_arr, keys):
    assert pico_arr.shape[0] == len(keys)

    gate_lookup = {}

    samples = pico_arr.shape[1]
    for k, row in zip(keys, pico_arr):
        signal_samples = ps.resample(
            duration,
            ps.get_samples(
                ps.custom_envelope(samples, row, dt=dt)
            )
        )

        gate_lookup[k] = ps.scalable_symbolic_pulse(duration, signal_samples, "custom")
        
    return gate_lookup

In [ ]:
gates = gateset_from_pico(duration, dt, pico_a, ["x1", "x2", "sx1", "sx2"])

In [ ]:
np.trapz(ps.get_samples(gates["x1"]).real, dx=dt) / np.pi

In [ ]:
samples = ps.get_samples(gates["x1"])

fig, ax = plt.subplots(figsize=(5, 3))
# Rescale to match the original length
scale = pico_a.shape[1] / duration
plt.plot(np.arange(0, duration) * scale, samples.real / scale, label="custom")
plt.plot(np.arange(pico_a.shape[1]), pico_a[0], label="pico", ls=':')
plt.legend()

In [ ]:
default_gates = {
    "x1": ps.gaussian_envelope(duration, dt=dt, angle=np.pi/2),
    "x2": ps.gaussian_envelope(duration, dt=dt, angle=np.pi/2),
    "sx1": ps.gaussian_envelope(duration, dt=dt, angle=np.pi/4),
    "sx2": ps.gaussian_envelope(duration, dt=dt, angle=np.pi/4),
}

In [ ]:
np.trapz(ps.get_samples(default_gates["x1"]).real, dx=dt) / np.pi

In [ ]:
qubit_sched = {0: "x1", 1: "x2"}
# qubit_sched = {0: "x1"}

before = simpson(ps.get_samples(gates[qubit_sched[0]]).real, dx=dt)
print(f"Before {before / np.pi} π")

with qk.pulse.build(name="Robust moment", backend=backend) as rabi_sched:
    for i, key in qubit_sched.items():
        channel = qk.pulse.DriveChannel(i)
        # TODO: Can we change the samples in the waveform returned by SymbolicPulse?
        qk.pulse.play(gates[key], channel)

with qk.pulse.build(name="Default moment", backend=backend) as default_sched:
    for i, key in qubit_sched.items():
        channel = qk.pulse.DriveChannel(i)
        # TODO: Can we change the samples in the waveform returned by SymbolicPulse?
        qk.pulse.play(default_gates[key], channel)

after = np.trapz(ps.get_samples(rabi_sched.instructions[0][1].pulse).real, dx=dt)
print(f"After {after / np.pi} π")

Two options: Circuit or pulse schedule.

In [ ]:
rabi_sched.draw()

In [ ]:
default_sched.draw()

## Simulate

In [ ]:
# Define models fixed by the current parameters

# Backend
# =====
registers = [0, 1]

# NOTE: If the Rabi rates are True, you have to calibrate!
config_vars = ps.backend_simulation_vars(backend, rabi=False, units=Units)

# Carrier frequencies of each control line
carriers = ps.backend_carriers(backend, config_vars)

# =====
H_rx = functools.partial(
    ps.rx_model, 
    registers=registers,
    backend=backend,
    variables=config_vars,
    rotating_frame=True,
)

A_decay = functools.partial(
    ps.qubit_decay_model,
    registers=registers, 
    variables=config_vars
)

H_xtalk = ps.crosstalk_model(
    registers,
    ps.backend_edges(backend),
    config_vars
)

# Create a system model
# =====
# Control model
#   Single qubit circuit moment: attach RX models to each qubit
H_drift = 0.
Hs_control = []
Hs_channels = []
As_static = []
for qubit, label in qubit_sched.items():
    
    # if qubit_index in registers:
    Hj_drift, Hjs_control, Hjs_channel = H_rx(qubit)

    # NOTE: doesn't work with quantum channel simulator yet
    Ajs_static = A_decay(qubit)

    H_drift += Hj_drift
    Hs_control += Hjs_control
    Hs_channels += Hjs_channel
    As_static += Ajs_static

In [ ]:
# Identity matrix
U0 = ps.qiskit_identity_operator(len(registers))

# Identity channel
C0 = qk.quantum_info.SuperOp(U0)

In [ ]:
xi_arr = np.linspace(0, 0.01, 20)
target = ps.from_label("XX")

H_xtalk_norm = H_xtalk / np.max(H_xtalk)

with_crosstalk = []
without_crosstalk = []
with_crosstalk_default = []
with_crosstalk_default = []
for xi in xi_arr:
    # Construct the solvers
    solver = qk_d.Solver(
        static_hamiltonian=xi * np.zeros_like(H_xtalk_norm),
        hamiltonian_operators=Hs_control,
        hamiltonian_channels=Hs_channels,
        channel_carrier_freqs={ch: 0. for ch in Hs_channels},
        dt=dt
    )

    solver_xtalk= qk_d.Solver(
        static_hamiltonian=xi * H_xtalk_norm,
        hamiltonian_operators=Hs_control,
        hamiltonian_channels=Hs_channels,
        channel_carrier_freqs={ch: 0. for ch in Hs_channels},
        dt=dt
    )

    # Rollout the pulses
    solver.model.evaluation_mode = 'sparse'
    sol = solver.solve(
        t_span=[0., duration], 
        y0=U0, 
        signals=rabi_sched,
        atol=1e-16,
        rtol=1e-16,
        method='jax_odeint'
    )
    sol_default = solver.solve(
        t_span=[0., duration], 
        y0=U0, 
        signals=default_sched,
        atol=1e-16,
        rtol=1e-16,
        method='jax_odeint'
    )

    without_crosstalk += [qk.quantum_info.process_fidelity(sol.y[-1], target)]
    without_crosstalk_default += [qk.quantum_info.process_fidelity(sol_default.y[-1], target)]

    solver_xtalk.model.evaluation_mode = 'sparse'
    sol_xtalk = solver_xtalk.solve(
        t_span=[0., duration], 
        y0=U0, 
        signals=rabi_sched,
        atol=1e-16,
        rtol=1e-16,
        method='jax_odeint'
    )
    sol_xtalk_default = solver_xtalk.solve(
        t_span=[0., duration], 
        y0=U0, 
        signals=default_sched,
        atol=1e-16,
        rtol=1e-16,
        method='jax_odeint'
    )

    with_crosstalk += [qk.quantum_info.process_fidelity(sol_xtalk.y[-1], target)]
    with_crosstalk_default += [qk.quantum_info.process_fidelity(sol_xtalk_default.y[-1], target)]


In [ ]:
without_crosstalk_default

In [ ]:
fig, ax = plt.subplots()
ax.plot(xi_arr, 1 - np.array(without_crosstalk), label='without, robust')
ax.plot(xi_arr, 1 - np.array(with_crosstalk), label='with, robust')

# ax.plot(xi_arr, 1 - np.array(without_crosstalk_default), label='without, default')
ax.plot(xi_arr, 1 - np.array(with_crosstalk_default), label='with, default')
ax.legend()
# ax.set_yscale('log')
# ax.set_xlim([0, 0.1])

Robustness is not preserved after resampling.

In [ ]:

# solver_decay = qk_d.Solver(
#     static_hamiltonian=np.zeros_like(H_xtalk),
#     hamiltonian_operators=Hs_control,
#     static_dissipators=As_static,
#     hamiltonian_channels=Hs_channels,
#     channel_carrier_freqs={ch: 0. for ch in Hs_channels},
#     dt=dt
# )

# solver_decay.model.evaluation_mode = 'sparse_vectorized'
# sol_decay = solver_decay.solve(
#     t_span=[0., duration], 
#     y0=C0, 
#     signals=rabi_sched,
#     atol=1e-16,
#     rtol=1e-16,
#     method='jax_odeint'
# )


# Calibrate

In [ ]:
# Experiment rabi schedules
drive_amp = qk.circuit.Parameter('drive_amp')
drive_amps = np.linspace(0.8, 1.2, 51)
exp_rabi_scheds = [rabi_sched.assign_parameters({drive_amp: a}, inplace=False) for a in drive_amps]

In [ ]:
# Add rabi gate to circuit
rabi_gate = qk.circuit.Gate("rabi", 1, [drive_amp])
qc_rabi = qk.QuantumCircuit(1, 1)
qc_rabi.append(rabi_gate, [0])
qc_rabi.measure(0, 0)

# Register a low-level, custom pulse definition for the given gate.
qc_rabi.add_calibration(rabi_gate, (0,), rabi_sched, [drive_amp])

# Execute rabi circuits
drive_amps = np.linspace(.8, 1.2, 51)
exp_rabi_circs = [qc_rabi.assign_parameters({drive_amp: a}, inplace=False) for a in drive_amps]

In [ ]:
rabi_schedule = qk.schedule(exp_rabi_circs[-1], backend)
rabi_schedule.draw(backend=backend)

To simulate, use fake 2 qubit?

In [ ]:
fake_2q = qk_fp.FakeOpenPulse2Q()

In [ ]:
num_shots_per_point = 1024

job = fake_2q.run(
    exp_rabi_circs, meas_level=1, meas_return="avg", shots=num_shots_per_point
)

# job.submit()

In [ ]:
# Fails because fake backends don't have pulse gate support
job.result()

# Miscellany

In [ ]:
# Check that resample works.
plt.step(np.linspace(0, duration, samples, endpoint=False), ps.resample(samples, gate_lookup[gate].samples).real)
plt.step(np.arange(0, duration), gate_lookup[gate].samples.real)